# Investment AI
## Notebook 04 — Multi-Format Research Document Ingestion & Parsing

The structured portfolio foundation is now ready.

This notebook starts the unstructured side of the platform by processing the 45 synthetic research documents stored in the Unity Catalog research Volume.

```text
Research Volume
│
├── PDF   ─┐
├── DOCX  ─┼─→ ai_parse_document → structured VARIANT
│          │
├── HTML  ─┐
└── TXT   ─┴─→ UTF-8 text extraction
                 ↓
        research.parsed_documents
```

The goal of this notebook is **document parsing**, not chunking.

Semantic chunking and search preparation come in the next notebook.


## Why use two parsing paths?

Databricks `ai_parse_document` understands document layout and produces structured elements such as:

- titles
- section headers
- text
- tables
- figures
- page references

That makes it appropriate for formats such as PDF and DOCX.

Our HTML and TXT sources are already text-based, so sending them through document parsing is unnecessary. We decode those files directly as UTF-8 text.

This gives us one governed table while preserving the best representation for each source type.


In [0]:
from pyspark.sql import functions as F

CATALOG = "investment_ai"
RESEARCH_SCHEMA = "research"

RAW_DOCUMENTS_PATH = (
    f"/Volumes/{CATALOG}/{RESEARCH_SCHEMA}/raw_documents/documents"
)

MANIFEST_TABLE = (
    f"{CATALOG}.{RESEARCH_SCHEMA}.document_manifest"
)

PARSED_TABLE = (
    f"{CATALOG}.{RESEARCH_SCHEMA}.parsed_documents"
)

print("Raw research documents :", RAW_DOCUMENTS_PATH)
print("Manifest table         :", MANIFEST_TABLE)
print("Parsed output table    :", PARSED_TABLE)


## 1. Confirm the parsing function is available

`ai_parse_document` requires a supported Databricks environment and AI Functions availability in the workspace region.

For serverless compute, the environment must support `VARIANT`.


In [0]:
available_functions = {
    row["function"]
    for row in spark.sql(
        "SHOW FUNCTIONS LIKE 'ai_parse_document'"
    ).collect()
}

assert "ai_parse_document" in available_functions, (
    "ai_parse_document is not available in this environment."
)

print("✅ ai_parse_document is available.")


## 2. Discover the landed research files

We read the research Volume using Spark's `binaryFile` data source.

This gives us:

- file path
- modification timestamp
- file length
- binary content

We also calculate a SHA-256 hash so the exact source bytes can be tracked.


In [0]:
source_files_df = (
    spark.read
    .format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load(RAW_DOCUMENTS_PATH)
    .withColumn(
        "file_name",
        F.element_at(F.split(F.col("path"), "/"), -1),
    )
    .withColumn(
        "file_extension",
        F.lower(
            F.regexp_extract(
                F.col("file_name"),
                r"\.([^.]+)$",
                1,
            )
        ),
    )
    .withColumn(
        "source_file_hash",
        F.sha2(F.col("content"), 256),
    )
)

print("Research files discovered:", source_files_df.count())
display(
    source_files_df.select(
        "file_name",
        "file_extension",
        "length",
        "modificationTime",
        "source_file_hash",
    )
)


## 3. Validate the research corpus

Our synthetic corpus contains:

```text
25 PDF
10 DOCX
 5 HTML
 5 TXT
-------
45 documents
```

We validate this before invoking an AI parsing function so that we do not spend compute on an incomplete or incorrect corpus.


In [0]:
extension_counts = {
    row["file_extension"]: row["count"]
    for row in (
        source_files_df
        .groupBy("file_extension")
        .count()
        .collect()
    )
}

expected_extension_counts = {
    "pdf": 25,
    "docx": 10,
    "html": 5,
    "txt": 5,
}

assert source_files_df.count() == 45
assert extension_counts == expected_extension_counts

for extension, count in sorted(extension_counts.items()):
    print(f"✅ {extension.upper():5s}: {count}")

print("\n✅ 45-document research corpus validated.")


## 4. Join the files to governed research metadata

Notebook 03 published the document manifest as a Unity Catalog table.

We join every physical source file to that governed metadata before parsing.

This gives each document stable business context such as:

- `document_id`
- synthetic `security_id`
- company
- sector
- fiscal quarter
- document type
- source type
- publication date


In [0]:
manifest_df = spark.table(MANIFEST_TABLE)

research_files_df = (
    source_files_df
    .join(
        manifest_df,
        on="file_name",
        how="left",
    )
)

unmatched_files = (
    research_files_df
    .filter(F.col("document_id").isNull())
    .count()
)

assert unmatched_files == 0, (
    f"{unmatched_files} research files do not match the document manifest."
)

assert research_files_df.select("document_id").distinct().count() == 45

print("✅ Every source file is linked to governed document metadata.")


## 5. Parse PDF and DOCX with `ai_parse_document`

PDF and DOCX contain layout information that is useful for downstream retrieval.

We use output schema version `2.0` explicitly rather than relying on a future default.

For this corpus, figure descriptions are disabled because they are not needed for the business question and would add unnecessary processing.

The result is stored as Databricks `VARIANT`.


In [0]:
layout_documents_df = (
    research_files_df
    .filter(
        F.col("file_extension").isin("pdf", "docx")
    )
    .withColumn(
        "parsed_content",
        F.expr(
            """
            ai_parse_document(
                content,
                map(
                    'version', '2.0',
                    'descriptionElementTypes', ''
                )
            )
            """
        ),
    )
    .withColumn(
        "text_content",
        F.lit(None).cast("string"),
    )
    .withColumn(
        "processing_method",
        F.lit("ai_parse_document_v2"),
    )
)

print(
    "Documents sent to ai_parse_document:",
    layout_documents_df.count(),
)


In [0]:
layout_documents_df.display()

## 6. Inspect parsing status

`ai_parse_document` returns an `error_status` array in the parsed result.

A successful document should have no parsing errors.

We inspect the status before publishing the data.


In [0]:
layout_validation_df = (
    layout_documents_df
    .withColumn(
        "parse_status",
        F.when(
            F.expr("is_variant_null(parsed_content:error_status)"),
            F.lit("SUCCESS"),
        ).otherwise(F.lit("REVIEW")),
    )
)

display(
    layout_validation_df.select(
        "document_id",
        "file_name",
        "file_extension",
        "parse_status",
        F.expr("parsed_content:error_status").alias("error_status"),
    )
)

documents_requiring_review = (
    layout_validation_df
    .filter(F.col("parse_status") != "SUCCESS")
    .count()
)

assert documents_requiring_review == 0, (
    f"{documents_requiring_review} parsed documents require review."
)

print("✅ All PDF and DOCX documents parsed successfully.")


## 7. Decode HTML and TXT sources directly

HTML and TXT are already text-based.

For HTML, we remove markup and normalize whitespace so the next stage receives readable text instead of HTML tags.

TXT is decoded directly as UTF-8.


In [0]:
decoded_text = F.decode(F.col("content"), "UTF-8")

text_documents_df = (
    research_files_df
    .filter(
        F.col("file_extension").isin("html", "txt")
    )
    .withColumn(
        "text_content",
        F.when(
            F.col("file_extension") == "html",
            F.trim(
                F.regexp_replace(
                    F.regexp_replace(
                        decoded_text,
                        r"(?s)<[^>]+>",
                        " ",
                    ),
                    r"\s+",
                    " ",
                )
            ),
        ).otherwise(decoded_text),
    )
    .withColumn(
        "parsed_content",
        F.expr("CAST(NULL AS VARIANT)"),
    )
    .withColumn(
        "processing_method",
        F.when(
            F.col("file_extension") == "html",
            F.lit("html_utf8_text"),
        ).otherwise(F.lit("txt_utf8_text")),
    )
)

empty_text_documents = (
    text_documents_df
    .filter(
        F.col("text_content").isNull()
        | (F.length(F.trim(F.col("text_content"))) == 0)
    )
    .count()
)

assert empty_text_documents == 0

print(
    "Text-based documents decoded:",
    text_documents_df.count(),
)
print("✅ HTML and TXT content extracted successfully.")


In [0]:
text_documents_df.display()

## 8. Build one governed parsed-document dataset

The table keeps both representations:

```text
PDF / DOCX
→ parsed_content  (VARIANT)

HTML / TXT
→ text_content    (STRING)
```

That distinction is intentional.

In the next notebook, `ai_prep_search` can work with the structured `VARIANT` representation as well as plain text.


In [0]:
output_columns = [
    "document_id",
    "file_name",
    "file_extension",
    "path",
    "source_file_hash",
    "length",
    "modificationTime",
    "security_id",
    "company_name",
    "sector",
    "fiscal_quarter",
    "fiscal_year",
    "document_type",
    "source_type",
    "source",
    "publish_date",
    "language",
    "confidentiality",
    "relative_path",
    "processing_method",
    "parsed_content",
    "text_content",
]

parsed_documents_df = (
    layout_documents_df.select(*output_columns)
    .unionByName(
        text_documents_df.select(*output_columns)
    )
)

assert parsed_documents_df.count() == 45
assert parsed_documents_df.select("document_id").distinct().count() == 45

print("✅ Unified parsed dataset contains 45 documents.")


## 9. Validate the representation by file type

We want to guarantee:

- PDF/DOCX → structured parsed content exists
- HTML/TXT → plain text content exists

This prevents partially processed documents from entering the retrieval pipeline.


In [0]:
invalid_layout_documents = (
    parsed_documents_df
    .filter(
        F.col("file_extension").isin("pdf", "docx")
        & F.col("parsed_content").isNull()
    )
    .count()
)

invalid_text_documents = (
    parsed_documents_df
    .filter(
        F.col("file_extension").isin("html", "txt")
        & (
            F.col("text_content").isNull()
            | (F.length(F.trim(F.col("text_content"))) == 0)
        )
    )
    .count()
)

assert invalid_layout_documents == 0
assert invalid_text_documents == 0

print("✅ PDF/DOCX structured representations validated.")
print("✅ HTML/TXT text representations validated.")


## 10. Publish `research.parsed_documents`

This is our governed handoff between document ingestion and semantic chunking.

We keep the raw source files in the Volume and publish the processed representation as a Delta table.


In [0]:
(
    parsed_documents_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PARSED_TABLE)
)

spark.sql(
    f"""
    COMMENT ON TABLE {PARSED_TABLE} IS
    'Governed parsed synthetic research documents for retrieval and agent workflows.'
    """
)

print(f"✅ Published {PARSED_TABLE}")


## 11. Inspect a structured parsed document

For PDF and DOCX, the parsed `VARIANT` contains elements such as titles, section headers, text, tables, figures, and page information.

We inspect one document here to make the representation visible before chunking it.


In [0]:
sample_layout_document = (
    spark.table(PARSED_TABLE)
    .filter(F.col("file_extension") == "pdf")
    .select(
        "document_id",
        "file_name",
        "company_name",
        "document_type",
        F.expr(
            "parsed_content:metadata"
        ).alias("parse_metadata"),
        F.expr(
            "parsed_content:document:elements"
        ).alias("document_elements"),
    )
    .limit(1)
)

display(sample_layout_document)


## 12. Inspect a directly decoded text document


In [0]:
display(
    spark.table(PARSED_TABLE)
    .filter(F.col("file_extension").isin("html", "txt"))
    .select(
        "document_id",
        "file_name",
        "company_name",
        "document_type",
        "processing_method",
        "text_content",
    )
    .limit(2)
)


## 13. Final document-ingestion validation

At the end of this notebook:

```text
45 source documents
      ↓
35 PDF/DOCX → structured document parsing
10 HTML/TXT → direct text extraction
      ↓
45 governed rows in research.parsed_documents
```

The raw files remain unchanged in the Unity Catalog Volume.

The next step is to transform these document representations into semantic, search-ready chunks.


In [0]:
final_df = spark.table(PARSED_TABLE)

final_counts = {
    row["file_extension"]: row["count"]
    for row in (
        final_df
        .groupBy("file_extension")
        .count()
        .collect()
    )
}

assert final_df.count() == 45
assert final_counts == {
    "pdf": 25,
    "docx": 10,
    "html": 5,
    "txt": 5,
}

assert (
    final_df
    .select("source_file_hash")
    .distinct()
    .count()
    == 45
)

print("✅ 45 documents published.")
print("✅ 35 PDF/DOCX documents parsed with ai_parse_document.")
print("✅ 10 HTML/TXT documents extracted as UTF-8 text.")
print("✅ Every document retains governed business metadata.")
print("✅ Source hashes provide file-level traceability.")
print("\nNext: 05_prepare_semantic_chunks.ipynb")
